# Accessing NOAA/NSIDC Sea Ice Concentration CDR V6 via STAC GeoParquet

This notebook demonstrates how to discover and access the NOAA/NSIDC Climate Data Record of Passive Microwave Sea Ice Concentration, Version 6 (G02202) using STAC GeoParquet.

Instead of crawling a STAC API, we load a GeoParquet file containing all STAC items directly. This approach is fast, works offline once the file is downloaded, and requires no special client beyond `pyarrow`.

**Citation:** Meier, W. N., F. Fetterer, A. K. Windnagel, J. S. Stewart, and T. Stafford. 2026. NOAA/NSIDC Climate Data Record of Passive Microwave Sea Ice Concentration, Version 6. Boulder, Colorado USA. NSIDC: National Snow and Ice Data Center. https://doi.org/10.7265/b18j-z797

## Setup

Install dependencies if needed:
```bash
pip install pyarrow stac-geoparquet pystac xarray fsspec requests h5netcdf
```

In [ ]:
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import stac_geoparquet.arrow as stac_geoparquet
import xarray as xr
import fsspec

## Load the STAC Items GeoParquet

Two GeoParquet files are available — one for daily items and one for monthly.
Each row is one STAC item (one file, one hemisphere, one day or month).

In [ ]:
STAC_BASE = "https://noaadata.apps.nsidc.org/NOAA/G02202_V6/stac/sea-ice-concentration"

daily_table = pq.read_table(f"{STAC_BASE}/sea-ice-concentration-daily-items.parquet")
monthly_table = pq.read_table(f"{STAC_BASE}/sea-ice-concentration-monthly-items.parquet")

print(f"Daily items:   {len(daily_table):,}")
print(f"Monthly items: {len(monthly_table):,}")
print(f"\nColumns: {daily_table.schema.names}")

## Filter Items

Use PyArrow compute functions to filter by date, hemisphere, or any other property.
This is fast even for the full archive because it operates on columnar data in memory.

In [ ]:
# Filter daily items: Northern Hemisphere, January 2024
mask = pc.and_(
    pc.and_(
        pc.greater_equal(daily_table["start_datetime"], pa.scalar("2024-01-01T00:00:00+00:00")),
        pc.less_equal(daily_table["start_datetime"], pa.scalar("2024-01-31T23:59:59+00:00")),
    ),
    pc.equal(daily_table["noaa_cdr:hemisphere"], "north"),
)
filtered = daily_table.filter(mask)
print(f"Found {len(filtered)} items")

## Convert Filtered Rows Back to STAC Items

Use `stac_geoparquet` to convert filtered rows back into STAC item dicts,
then extract asset URLs for data access.

In [ ]:
items = list(stac_geoparquet.stac_table_to_items(filtered))

# Look at the first item
item = items[0]
print("Item ID:", item["id"])
print("Start:", item["properties"]["start_datetime"])
print("Hemisphere:", item["properties"]["noaa_cdr:hemisphere"])
print("\nAssets:")
for key, asset in item["assets"].items():
    print(f"  {key}: {asset['href']}")

## Open a NetCDF File Directly

Get the NetCDF URL from the item's assets and open it in-place with `fsspec` + `xarray`.
No download required.

In [ ]:
netcdf_url = item["assets"]["netcdf"]["href"]
print("Opening:", netcdf_url)

fs = fsspec.filesystem("https")
ds = xr.open_dataset(fs.open(netcdf_url), engine="h5netcdf")
ds

## Extract Sea Ice Concentration

The primary variable is `cdr_seaice_conc`. Values are scaled (multiply by `scale_factor` to get fractional concentration 0–1). The `surface_type_mask` variable (in the `cdr_supplementary` group) identifies ocean, land, coast, lake, and pole hole pixels.

In [ ]:
sic = ds["cdr_seaice_conc"]
print(sic)
print("\nScale factor:", sic.attrs.get("scale_factor"))
print("Units:", sic.attrs.get("units"))

## Quick Plot

In [ ]:
import matplotlib.pyplot as plt

# Apply scale factor and squeeze time dimension
conc = sic.values.squeeze() * sic.attrs.get("scale_factor", 1.0)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(conc, cmap="Blues_r", vmin=0, vmax=1, origin="upper")
plt.colorbar(im, ax=ax, label="Sea Ice Concentration")
ax.set_title(f"Sea Ice Concentration\n{item['id']}")
ax.axis("off")
plt.tight_layout()
plt.show()

## Working with Monthly Data

The same pattern works for monthly items. Monthly files contain `cdr_seaice_conc_monthly` as the primary variable.

In [ ]:
# Filter monthly: Southern Hemisphere, 2023
mask_monthly = pc.and_(
    pc.and_(
        pc.greater_equal(monthly_table["start_datetime"], pa.scalar("2023-01-01T00:00:00+00:00")),
        pc.less_equal(monthly_table["start_datetime"], pa.scalar("2023-12-31T23:59:59+00:00")),
    ),
    pc.equal(monthly_table["noaa_cdr:hemisphere"], "south"),
)
monthly_filtered = monthly_table.filter(mask_monthly)
monthly_items = list(stac_geoparquet.stac_table_to_items(monthly_filtered))
print(f"Found {len(monthly_items)} monthly items for Southern Hemisphere 2023")

# Open the first one
monthly_item = monthly_items[0]
monthly_url = monthly_item["assets"]["netcdf"]["href"]
print("URL:", monthly_url)

ds_monthly = xr.open_dataset(fs.open(monthly_url), engine="h5netcdf")
print(ds_monthly["cdr_seaice_conc_monthly"])

## Data Statement

Meier, W. N., F. Fetterer, A. K. Windnagel, J. S. Stewart, and T. Stafford. 2026. NOAA/NSIDC Climate Data Record of Passive Microwave Sea Ice Concentration, Version 6. Boulder, Colorado USA. NSIDC: National Snow and Ice Data Center. https://doi.org/10.7265/b18j-z797

## Software License

Copyright 2026 NOAA NCAI. Licensed under the MIT License.